# Compare loss functions — Tweedie vs Gamma vs Quantile

5-fold CV with **identical fold splits** as `lgbm_train.ipynb` (loaded from `s3://thesis-data-ismaktam/lgbm/fold_assignment.parquet`). Re-uses the **same per-fold geo-features** computed by the quantile run (`s3://thesis-data-ismaktam/lgbm/kfold/foldI_features.parquet`) — no recomputation.

Trains:
- **Tweedie** (variance_power=1.5, point prediction)
- **Gamma** (point prediction, requires y > 0)

**No final model**, CV only. Multi-GPU: each fold trains both losses in parallel across GPUs (2 jobs per fold).

## Metrics

Tweedie and Gamma are **point predictors** — CRPS is undefined without a distributional assumption, so we report only point metrics:

| Loss     | MAE | RMSE | CRPS |
|---       |---  |---   |---   |
| Tweedie  | ✓   | ✓    | n/a (point predictor) |
| Gamma    | ✓   | ✓    | n/a (point predictor) |
| Quantile | ✓ (median) | ✓ (median) | true `crps_ensemble` from 11 quantiles (loaded from quantile run) |

The CRPS comparison is therefore between the quantile baseline and itself across folds; the Tweedie/Gamma vs quantile comparison is on MAE/RMSE only.

Output: `s3://thesis-data-ismaktam/lgbm/compare_loss/cv_results.json`

In [1]:
import sys, os, gc, json, warnings
import subprocess, tempfile, shutil, textwrap
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib
import boto3

warnings.filterwarnings('ignore')

ROOT = Path('../..')
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

print(f'Working directory: {Path.cwd()}')
print(f'LightGBM: {lgb.__version__}')

Working directory: /root/precip_interpolation_thesis
LightGBM: 4.5.0


In [2]:
# ── Reproducibility — MUST match lgbm_train.ipynb ────────────────────────────
RANDOM_SEED = 42
K_FOLDS     = 5

# ── Geo-features (must match lgbm_train.ipynb so the parquets we download are valid) ──
K_GEO         = 15
SVD_QUANTILES = np.arange(0.0, 1.05, 0.05)
SVD_COLS      = [f'svd_{i:02d}' for i in range(len(SVD_QUANTILES))]

# ── LightGBM ─────────────────────────────────────────────────────────────────
NUM_BOOST_ROUND = 1000
EARLY_STOPPING  = 50
DEVICE          = 'cuda'   # 'cuda' on Vast.ai  |  'cpu' locally

# ── Loss-function configs ────────────────────────────────────────────────────
# Same shared HPO-best tree shape as the quantile run, only the objective changes.
# CRITICAL: `metric` lists the NATIVE objective metric FIRST so that early-stopping
# (with first_metric_only=True) watches the same loss the model is actually
# minimising. Putting `l1` first would early-stop after ~10 rounds because L1 on
# point predictions plateaus quickly while the native deviance keeps improving.
LGB_SHARED = dict(
    learning_rate     = 0.05,
    max_depth         = 10,
    num_leaves        = 200,
    min_child_samples = 100,
    feature_fraction  = 0.8,
    bagging_fraction  = 0.8,
    bagging_freq      = 1,
    device_type       = DEVICE,
    num_threads       = -1,
    seed              = RANDOM_SEED,
    verbose           = -1,
)

LOSS_CONFIGS = {
    'tweedie': {
        **LGB_SHARED,
        'objective':              'tweedie',
        'tweedie_variance_power': 1.5,
        'metric':                 ['tweedie', 'l1'],
    },
    'gamma': {
        **LGB_SHARED,
        'objective': 'gamma',
        'metric':    ['gamma', 'l1'],
    },
}
LOSS_NAMES = list(LOSS_CONFIGS.keys())

# ── GPU detection ────────────────────────────────────────────────────────────
def _detect_gpus():
    try:
        out = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True, check=True,
        )
        rows = [r.strip().split(', ') for r in out.stdout.strip().split('\n') if r.strip()]
        if rows:
            return len(rows), float(rows[0][1]) / 1024.0
    except Exception:
        pass
    return 0, 0.0

_n_detected, GPU_VRAM_GB = _detect_gpus()
N_GPUS       = _n_detected if DEVICE == 'cuda' and _n_detected > 0 else 1
N_PARALLEL   = min(N_GPUS, len(LOSS_NAMES))

# ── Paths ────────────────────────────────────────────────────────────────────
OUT_DIR    = Path('outputs/lgbm/compare_loss')
OUT_DIR.mkdir(parents=True, exist_ok=True)

S3_BUCKET    = 'thesis-data-ismaktam'
S3_LGBM_ROOT = 'lgbm'
S3_OUT_ROOT  = f'{S3_LGBM_ROOT}/compare_loss'

FORCE_RECOMPUTE = False

print(f'RANDOM_SEED={RANDOM_SEED}  K_FOLDS={K_FOLDS}  DEVICE={DEVICE}')
print(f'NUM_BOOST_ROUND={NUM_BOOST_ROUND}  EARLY_STOPPING={EARLY_STOPPING}')
print(f'Loss configs: {LOSS_NAMES}')
if DEVICE == 'cuda':
    print(f'GPUs detected: {_n_detected}  (VRAM: {GPU_VRAM_GB:.1f} GB / GPU)')
    print(f'Per-fold parallelism: {len(LOSS_NAMES)} losses across {N_PARALLEL} GPU(s)')
else:
    print('CPU mode — sequential')
print(f'S3 output root: s3://{S3_BUCKET}/{S3_OUT_ROOT}/')

RANDOM_SEED=42  K_FOLDS=5  DEVICE=cuda
NUM_BOOST_ROUND=1000  EARLY_STOPPING=50
Loss configs: ['tweedie', 'gamma']
GPUs detected: 3  (VRAM: 15.9 GB / GPU)
Per-fold parallelism: 2 losses across 2 GPU(s)
S3 output root: s3://thesis-data-ismaktam/lgbm/compare_loss/


In [3]:
# ── GPU worker module — trains one (loss_name, params) on one GPU ──────────────
_WORKER_DIR = OUT_DIR / '_gpu_worker'
_WORKER_DIR.mkdir(parents=True, exist_ok=True)
_WORKER_PY = _WORKER_DIR / '_lgbm_loss_worker.py'
_WORKER_PY.write_text(textwrap.dedent("""\
    import os, gc
    import numpy as np
    import lightgbm as lgb


    def run_gpu_loss(args):
        (gpu_id, loss_name, params, data_dir,
         num_boost_round, early_stopping, has_val) = args

        if gpu_id >= 0:
            os.environ['CUDA_VISIBLE_DEVICES'] = str(gpu_id)

        params = dict(params)
        if params.get('device_type') == 'cuda':
            params['gpu_device_id'] = 0

        X_tr = np.load(f'{data_dir}/X_tr.npy', mmap_mode='r')
        y_tr = np.load(f'{data_dir}/y_tr.npy', mmap_mode='r')
        X_va = y_va = None
        if has_val:
            X_va = np.load(f'{data_dir}/X_va.npy', mmap_mode='r')
            y_va = np.load(f'{data_dir}/y_va.npy', mmap_mode='r')

        # Gamma requires y > 0. df_wet should already enforce this, but defensively clip.
        if loss_name == 'gamma':
            y_tr = np.maximum(np.asarray(y_tr), 1e-3).astype(np.float32)

        dtrain = lgb.Dataset(np.asarray(X_tr), label=np.asarray(y_tr), free_raw_data=False)
        dval   = (lgb.Dataset(np.asarray(X_va), label=np.asarray(y_va),
                              reference=dtrain, free_raw_data=False)
                  if has_val else None)

        if has_val:
            booster = lgb.train(
                params, dtrain,
                num_boost_round=num_boost_round,
                valid_sets=[dval],
                callbacks=[
                    # first_metric_only=True → watch the FIRST entry in `metric`
                    # (the native deviance, e.g. 'tweedie' or 'gamma'); l1 is
                    # logged but does not drive stopping.
                    lgb.early_stopping(early_stopping, first_metric_only=True, verbose=False),
                    lgb.log_evaluation(period=-1),
                ],
            )
            preds_va  = np.clip(booster.predict(np.asarray(X_va)), 0.0, None).astype(np.float32)
            best_iter = int(booster.best_iteration)
        else:
            booster = lgb.train(
                params, dtrain,
                num_boost_round=num_boost_round,
                callbacks=[lgb.log_evaluation(period=-1)],
            )
            preds_va  = None
            best_iter = int(booster.num_trees())

        result = {
            'loss_name': loss_name,
            'best_iter': best_iter,
            'preds_va':  preds_va,
            'model_str': booster.model_to_string(),
        }
        print(f'GPU{gpu_id} loss={loss_name:8s} done (best_iter={best_iter})', flush=True)
        del booster
        gc.collect()
        return result
"""))

if str(_WORKER_DIR) not in sys.path:
    sys.path.insert(0, str(_WORKER_DIR))
from _lgbm_loss_worker import run_gpu_loss   # noqa: E402,F401


def train_losses_multigpu(X_tr, y_tr, X_va, y_va,
                          loss_configs=LOSS_CONFIGS,
                          num_boost_round=NUM_BOOST_ROUND,
                          early_stopping=EARLY_STOPPING,
                          label='train'):
    has_val   = X_va is not None and y_va is not None
    n_workers = max(1, min(N_PARALLEL, len(loss_configs)))

    tmp_dir = tempfile.mkdtemp(prefix=f'lgbm_loss_{label}_')
    try:
        np.save(f'{tmp_dir}/X_tr.npy', X_tr)
        np.save(f'{tmp_dir}/y_tr.npy', y_tr)
        if has_val:
            np.save(f'{tmp_dir}/X_va.npy', X_va)
            np.save(f'{tmp_dir}/y_va.npy', y_va)

        loss_items  = list(loss_configs.items())
        worker_args = [
            (i if DEVICE == 'cuda' else -1,
             name, params, tmp_dir,
             num_boost_round, early_stopping, has_val)
            for i, (name, params) in enumerate(loss_items)
        ]

        if n_workers == 1:
            results_list = [run_gpu_loss(a) for a in worker_args]
        else:
            ctx = mp.get_context('spawn')
            with ProcessPoolExecutor(max_workers=n_workers, mp_context=ctx) as pool:
                futures = {pool.submit(run_gpu_loss, a): a[1] for a in worker_args}
                results_list = [fut.result() for fut in as_completed(futures)]
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    return {
        r['loss_name']: {
            'preds_va':  r['preds_va'],
            'best_iter': r['best_iter'],
            'model':     lgb.Booster(model_str=r['model_str']),
        }
        for r in results_list
    }

In [4]:
# ── Load station data + soil  (mirrors lgbm_train.ipynb cell-02-data) ────────
from thesis.config import Config
from thesis.data.registry import DataRegistry
from thesis.transforms import ProjectionTransform, IndicatorTransform
from thesis.transforms.pipeline import TransformPipeline
from thesis.scripts.run_grk_kfold_cv import SOIL_VARS

cfg      = Config()
registry = DataRegistry.from_config(cfg)

print(f'Date range: {cfg.date_start} → {cfg.date_end}')
print('Loading station data (exclude_holdout=True)...')

all_raw  = registry.stations.load(cfg.date_start, cfg.date_end)
pipeline = TransformPipeline([
    ProjectionTransform(target_crs=cfg.study_area.target_crs),
    IndicatorTransform(threshold_mm=cfg.wet_day_threshold_mm),
])
all_proc = pipeline.fit_transform(all_raw)
df_wet   = all_proc[all_proc['rain_indicator'] == 1].copy()

station_meta = (
    all_proc
    .drop_duplicates('station_id')[['station_id', 'x_proj', 'y_proj', 'elevation_m']]
    .reset_index(drop=True)
)

print(f'Wet records:  {len(df_wet):,}')
print(f'Stations:     {len(station_meta):,}')

soil_rows = {'station_id': station_meta['station_id'].values}
for var, src in registry.soilgrids.items():
    if var in SOIL_VARS:
        soil_rows[var] = src.sample_at_projected(
            station_meta['x_proj'].values,
            station_meta['y_proj'].values,
        )
soil_static    = pd.DataFrame(soil_rows).set_index('station_id')
available_soil = [v for v in SOIL_VARS if v in soil_static.columns]
for v in available_soil:
    soil_static[v] = soil_static[v].fillna(float(soil_static[v].median()))

FEATURE_COLS = (
    ['x_proj', 'y_proj', 'elevation_m']
    + ['idw', 'gos']
    + SVD_COLS
    + available_soil
)
TARGET_COL = 'precip_mm'

print(f'SoilGrids vars: {available_soil}')
print(f'Total features: {len(FEATURE_COLS)}')

Date range: 1961-01-01 → 2023-12-31
Loading station data (exclude_holdout=True)...
Wet records:  17,409,376
Stations:     1,966
SoilGrids vars: ['bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa']
Total features: 32


In [5]:
# ── S3 helpers ───────────────────────────────────────────────────────────────
s3 = boto3.client('s3')

def s3_download(s3_key: str, local_path: Path) -> bool:
    try:
        local_path.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(S3_BUCKET, s3_key, str(local_path))
        print(f'  ↓ s3://{S3_BUCKET}/{s3_key}')
        return True
    except Exception as e:
        print(f'  S3 download failed: {e.__class__.__name__}')
        return False

def s3_upload(local_path: Path, s3_key: str) -> None:
    try:
        s3.upload_file(str(local_path), S3_BUCKET, s3_key)
        print(f'  ↑ s3://{S3_BUCKET}/{s3_key}')
    except Exception as e:
        print(f'  S3 upload skipped: {e.__class__.__name__}: {e}')

# ── Fold assignment — MUST come from same S3 file as the quantile CV ─────────
FOLD_LOCAL = Path('outputs/lgbm/fold_assignment.parquet')
FOLD_S3    = f'{S3_LGBM_ROOT}/fold_assignment.parquet'

if not FOLD_LOCAL.exists() or FORCE_RECOMPUTE:
    ok = s3_download(FOLD_S3, FOLD_LOCAL)
    if not ok:
        raise RuntimeError(
            f'fold_assignment.parquet not in S3 — run lgbm_train.ipynb first '
            f'so the quantile CV creates and uploads it.'
        )
df_folds = pd.read_parquet(FOLD_LOCAL)
print(f'Fold assignment: {len(df_folds):,} stations across {df_folds.fold.nunique()} folds')
print(df_folds['fold'].value_counts().sort_index().to_string())

station_meta = station_meta.merge(df_folds[['station_id', 'fold']], on='station_id', how='left')
df_wet       = df_wet.merge(df_folds[['station_id', 'fold']], on='station_id', how='left')
assert df_wet['fold'].isna().sum() == 0, 'Some wet records have no fold assignment!'

Fold assignment: 1,966 stations across 5 folds
fold
0    394
1    393
2    393
3    393
4    393


In [6]:
# ── CV loop — for each fold, download geo-features + train both losses on GPUs ──
CV_RESULTS_LOCAL = OUT_DIR / 'cv_results.json'
CV_RESULTS_S3    = f'{S3_OUT_ROOT}/cv_results.json'

if CV_RESULTS_LOCAL.exists() and not FORCE_RECOMPUTE:
    cv_results = json.load(open(CV_RESULTS_LOCAL))
    done_folds = {r['fold'] for r in cv_results}
    print(f'Resuming — folds done: {sorted(done_folds)}')
else:
    cv_results = []
    done_folds = set()

for fold_id in range(K_FOLDS):
    if fold_id in done_folds:
        print(f'Fold {fold_id}: checkpoint found, skipping')
        continue

    print(f'\n{"="*60}')
    print(f'FOLD {fold_id} / {K_FOLDS - 1}')
    print(f'{"="*60}')

    feat_local = Path(f'outputs/lgbm/kfold/fold{fold_id}_features.parquet')
    feat_s3    = f'{S3_LGBM_ROOT}/kfold/fold{fold_id}_features.parquet'
    if not feat_local.exists() or FORCE_RECOMPUTE:
        ok = s3_download(feat_s3, feat_local)
        if not ok:
            raise RuntimeError(
                f'fold{fold_id}_features.parquet not in S3 — run lgbm_train.ipynb first '
                f'to compute geo-features for this fold.'
            )
    else:
        print(f'  Features: local cache')

    df_geo  = pd.read_parquet(feat_local)
    df_fold = (
        df_wet[['station_id', 'date', 'precip_mm', 'fold',
                'x_proj', 'y_proj', 'elevation_m']]
        .merge(df_geo, on=['station_id', 'date'], how='inner')
        .merge(soil_static[available_soil].reset_index(), on='station_id', how='left')
    )
    for v in available_soil:
        df_fold[v] = df_fold[v].fillna(df_fold[v].median())

    train_mask = df_fold['fold'] != fold_id
    test_mask  = df_fold['fold'] == fold_id

    X_tr = df_fold.loc[train_mask, FEATURE_COLS].values.astype(np.float32)
    y_tr = df_fold.loc[train_mask, TARGET_COL].values.astype(np.float32)
    X_te = df_fold.loc[test_mask,  FEATURE_COLS].values.astype(np.float32)
    y_te = df_fold.loc[test_mask,  TARGET_COL].values.astype(np.float32)

    n_tr_st = df_fold.loc[train_mask, 'station_id'].nunique()
    n_te_st = df_fold.loc[test_mask,  'station_id'].nunique()
    print(f'  Train: {X_tr.shape}  ({n_tr_st} stations)')
    print(f'  Test:  {X_te.shape}  ({n_te_st} stations)')
    del df_geo, df_fold; gc.collect()

    print(f'  Training {len(LOSS_NAMES)} losses on {N_PARALLEL} GPU(s)...')
    fold_models = train_losses_multigpu(
        X_tr, y_tr, X_te, y_te,
        loss_configs    = LOSS_CONFIGS,
        num_boost_round = NUM_BOOST_ROUND,
        early_stopping  = EARLY_STOPPING,
        label           = f'fold{fold_id}',
    )

    # ── Metrics — point only (Tweedie + Gamma both lack a built-in CRPS) ──────
    fold_record = {'fold': fold_id, 'n_test': int(len(y_te)), 'n_test_stations': int(n_te_st)}

    for loss_name, info in fold_models.items():
        y_pred_te = info['preds_va']
        mae  = float(np.abs(y_te - y_pred_te).mean())
        rmse = float(np.sqrt(((y_te - y_pred_te) ** 2).mean()))
        fold_record[loss_name] = {
            'mae':       mae,
            'rmse':      rmse,
            'best_iter': info['best_iter'],
        }
        print(f'  [{loss_name:7s}] MAE={mae:.4f}  RMSE={rmse:.4f}  '
              f'best_iter={info["best_iter"]}')

    # ── Save models + checkpoint ──────────────────────────────────────────────
    mdl_local = OUT_DIR / f'fold{fold_id}_models.joblib'
    mdl_s3    = f'{S3_OUT_ROOT}/fold{fold_id}_models.joblib'
    joblib.dump({
        'models':          {n: m['model']     for n, m in fold_models.items()},
        'best_iterations': {n: m['best_iter'] for n, m in fold_models.items()},
        'feature_cols':    FEATURE_COLS,
        'random_seed':     RANDOM_SEED,
    }, mdl_local)
    s3_upload(mdl_local, mdl_s3)

    cv_results.append(fold_record)
    json.dump(cv_results, open(CV_RESULTS_LOCAL, 'w'), indent=2)
    s3_upload(CV_RESULTS_LOCAL, CV_RESULTS_S3)

    del X_tr, y_tr, X_te, y_te, fold_models; gc.collect()
    print(f'Fold {fold_id} complete.')


FOLD 0 / 4
  Features: local cache
  Train: (13885714, 32)  (1572 stations)
  Test:  (3520827, 32)  (394 stations)
  Training 2 losses on 2 GPU(s)...
GPU1 loss=gamma    done (best_iter=10)
GPU0 loss=tweedie  done (best_iter=23)
  [gamma  ] MAE=2.3985  RMSE=4.9329  best_iter=10
  [tweedie] MAE=1.7323  RMSE=3.5276  best_iter=23
  ↑ s3://thesis-data-ismaktam/lgbm/compare_loss/fold0_models.joblib
  ↑ s3://thesis-data-ismaktam/lgbm/compare_loss/cv_results.json
Fold 0 complete.

FOLD 1 / 4
  Features: local cache
  Train: (13940928, 32)  (1573 stations)
  Test:  (3466145, 32)  (393 stations)
  Training 2 losses on 2 GPU(s)...
GPU1 loss=gamma    done (best_iter=10)
GPU0 loss=tweedie  done (best_iter=23)
  [gamma  ] MAE=2.3671  RMSE=4.8559  best_iter=10
  [tweedie] MAE=1.7117  RMSE=3.4713  best_iter=23
  ↑ s3://thesis-data-ismaktam/lgbm/compare_loss/fold1_models.joblib
  ↑ s3://thesis-data-ismaktam/lgbm/compare_loss/cv_results.json
Fold 1 complete.

FOLD 2 / 4
  Features: local cache
  Train:

In [7]:
# ── Summary table — Tweedie vs Gamma vs Quantile on the same folds ───────────
cv_results = json.load(open(CV_RESULTS_LOCAL))

rows = []
for r in cv_results:
    for loss_name in LOSS_NAMES:
        m = r[loss_name]
        rows.append({
            'fold':      r['fold'],
            'loss':      loss_name,
            'mae':       m['mae'],
            'rmse':      m['rmse'],
            'best_iter': m['best_iter'],
            'crps':      np.nan,   # point predictor — CRPS not defined
        })
df_loss = pd.DataFrame(rows)

# Quantile baseline from S3 — has true CRPS_ensemble from 11 quantiles
QUANTILE_CV_S3    = f'{S3_LGBM_ROOT}/kfold/cv_results.json'
QUANTILE_CV_LOCAL = Path('outputs/lgbm/kfold/cv_results.json')
if not QUANTILE_CV_LOCAL.exists():
    s3_download(QUANTILE_CV_S3, QUANTILE_CV_LOCAL)

if QUANTILE_CV_LOCAL.exists():
    qres = json.load(open(QUANTILE_CV_LOCAL))
    qrows = [{'fold': r['fold'], 'loss': 'quantile',
              'crps': r['crps'], 'mae': r['mae'], 'rmse': r['rmse'],
              'best_iter': None} for r in qres]
    df_loss = pd.concat([df_loss, pd.DataFrame(qrows)], ignore_index=True)

print('=== Per-fold MAE (point prediction) ===')
pivot_mae = df_loss.pivot(index='fold', columns='loss', values='mae').round(4)
print(pivot_mae.to_string())
print()

print('=== Per-fold RMSE ===')
pivot_rmse = df_loss.pivot(index='fold', columns='loss', values='rmse').round(4)
print(pivot_rmse.to_string())
print()

if 'quantile' in df_loss['loss'].values:
    print('=== Per-fold CRPS (quantile only — Tweedie/Gamma are point predictors) ===')
    pivot_crps = df_loss[df_loss['loss']=='quantile'].pivot(
        index='fold', columns='loss', values='crps').round(4)
    print(pivot_crps.to_string())
    print()

print('=== Mean ± std across folds ===')
for metric in ('mae', 'rmse'):
    print(f'\n{metric.upper()}:')
    agg = df_loss.groupby('loss')[metric].agg(['mean', 'std']).round(4)
    print(agg.to_string())

if 'quantile' in df_loss['loss'].values:
    print()
    print('=== Improvement of quantile over each loss (point metrics only) ===')
    q_mae  = df_loss[df_loss['loss']=='quantile']['mae'].mean()
    q_rmse = df_loss[df_loss['loss']=='quantile']['rmse'].mean()
    for loss_name in LOSS_NAMES:
        sub = df_loss[df_loss['loss']==loss_name]
        l_mae  = sub['mae'].mean()
        l_rmse = sub['rmse'].mean()
        print(f'  quantile vs {loss_name}:')
        print(f'    MAE:  {l_mae:.4f} → {q_mae:.4f}  ({(l_mae - q_mae)/l_mae*100:+.1f}%)')
        print(f'    RMSE: {l_rmse:.4f} → {q_rmse:.4f}  ({(l_rmse - q_rmse)/l_rmse*100:+.1f}%)')

=== Per-fold MAE (point prediction) ===
loss   gamma  quantile  tweedie
fold                           
0     2.3985    1.0663   1.7323
1     2.3671    1.0556   1.7117
2     2.3772    1.0400   1.7141
3     2.3513    1.0228   1.7399
4     2.3726    1.0239   1.7054

=== Per-fold RMSE ===
loss   gamma  quantile  tweedie
fold                           
0     4.9329    2.1641   3.5276
1     4.8559    2.1409   3.4713
2     4.8646    2.1097   3.4673
3     4.8368    2.0928   3.5272
4     4.8994    2.1095   3.4936

=== Per-fold CRPS (quantile only — Tweedie/Gamma are point predictors) ===
loss  quantile
fold          
0       0.8039
1       0.7962
2       0.7871
3       0.7742
4       0.7769

=== Mean ± std across folds ===

MAE:
            mean     std
loss                    
gamma     2.3733  0.0171
quantile  1.0417  0.0192
tweedie   1.7207  0.0147

RMSE:
            mean     std
loss                    
gamma     4.8779  0.0382
quantile  2.1234  0.0286
tweedie   3.4974  0.0291

=== Improve